In [1]:
import pandas as pd

# Load your file
df = pd.read_csv('7-features-combined-2783.csv')

# Drop the column
df = df.drop(columns=['cough_path'])

# Save the updated file
df.to_csv('7-features-combined-2783-updated.csv', index=False)

In [3]:
import pandas as pd
import numpy as np
import librosa
import parselmouth
from parselmouth.praat import call
from tqdm import tqdm
import os

In [4]:
print(df.shape)
print(df.columns)
df.head()

(2783, 9)
Index(['id', 'age', 'gender', 'smoker', 'cold_present', 'cough_present',
       'fever_present', 'disease', 'vowel_path'],
      dtype='object')


,id,age,gender,smoker,cold_present,cough_present,fever_present,disease,vowel_path
0,iV3Db6t1T8b7c5HQY2TwxIhjbzD3,28,male,NaN,False,False,False,healthy,E:\data\coswara-github\. Extracted_data\202004...
1,AxuYWBN0jFVLINCBqIW5aZmGCdu1,25,male,True,False,False,False,healthy,E:\data\coswara-github\. Extracted_data\202004...
2,C5eIsssb9GSkaAgIfsHMHeR6fSh1,28,female,NaN,False,False,False,healthy,E:\data\coswara-github\. Extracted_data\202004...
3,YjbEAECMBIaZKyfqOvWy5DDImUb2,26,male,NaN,True,False,False,healthy,E:\data\coswara-github\. Extracted_data\202004...
4,aGOvk4ji0cVqIzCs1jHnzlw2UEy2,32,male,NaN,False,False,False,healthy,E:\data\coswara-github\. Extracted_data\202004...


In [ ]:
AUDIO_COL = "vowel_path"   
LABEL_COL = "disease"

In [38]:
#Path rebuilding

import os
import re

COSWARA_BASE = r"D:\respiratory_classification_project\coswara-data"
KAGGLE_BASE = r"D:\respiratory_classification_project\kaggle-data"

def rebuild_vowel_path(old_path):
    old_path = str(old_path).replace("/", "\\")

    # Coswara format: ...\20200413\user_id\vowel-o.wav
    if "vowel-o.wav" in old_path:
        match = re.search(r"(20\d{6})\\([^\\]+)\\vowel-o\.wav", old_path)
        if match:
            date_folder = match.group(1)
            user_id = match.group(2)
            return os.path.join(COSWARA_BASE, date_folder, user_id, "vowel-o.wav")

    # Kaggle format: ...\recording_id\vowel.wav
    if "vowel.wav" in old_path:
        recording_id = os.path.basename(os.path.dirname(old_path))
        return os.path.join(KAGGLE_BASE, recording_id, "vowel.wav")

    return old_path

df[AUDIO_COL] = df[AUDIO_COL].apply(rebuild_vowel_path)

for i in range(10):
    path = df[AUDIO_COL].iloc[i]
    print(path)
    print("Exists:", os.path.exists(path))
    print("-" * 60)


D:\respiratory_classification_project\coswara-data\20200424\AxuYWBN0jFVLINCBqIW5aZmGCdu1\vowel-o.wav
Exists: True
------------------------------------------------------------
D:\respiratory_classification_project\coswara-data\20200424\C5eIsssb9GSkaAgIfsHMHeR6fSh1\vowel-o.wav
Exists: True
------------------------------------------------------------
D:\respiratory_classification_project\coswara-data\20200424\YjbEAECMBIaZKyfqOvWy5DDImUb2\vowel-o.wav
Exists: True
------------------------------------------------------------
D:\respiratory_classification_project\coswara-data\20200424\aGOvk4ji0cVqIzCs1jHnzlw2UEy2\vowel-o.wav
Exists: True
------------------------------------------------------------
D:\respiratory_classification_project\coswara-data\20200424\yWp5tMRFDzbbeEe2csKNd909fqh1\vowel-o.wav
Exists: True
------------------------------------------------------------
D:\respiratory_classification_project\coswara-data\20200424\rB5oGtrGYZR5uJUXEaDYrrredz13\vowel-o.wav
Exists: True
-----------

In [39]:
#Dropping missing files
df = df.dropna(subset=[AUDIO_COL]).reset_index(drop=True)

df[AUDIO_COL] = df[AUDIO_COL].astype(str)

df = df[df[AUDIO_COL].apply(os.path.exists)].reset_index(drop=True)

print("Files to process:", len(df))

#Check for missing files
missing = df[~df[AUDIO_COL].apply(os.path.exists)]

print("Total files:", len(df))
print("Missing files:", len(missing))
print("Found files:", len(df) - len(missing))


missing[[AUDIO_COL]].head(20)

Files to process: 2777
Total files: 2777
Missing files: 0
Found files: 2777


,vowel_path


In [40]:
import numpy as np
import librosa
import parselmouth
from parselmouth.praat import call

def extract_vowel_features(file_path, sr=16000, max_duration=5):
    features = {}

    try:
        # ---------- Librosa features ----------
        y, sr = librosa.load(file_path, sr=sr, duration=max_duration)
        y, _ = librosa.effects.trim(y, top_db=30)

        if len(y) == 0:
            return None

        features["duration"] = librosa.get_duration(y=y, sr=sr)

        # MFCC: 13 mean + 13 std = 26
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        for i in range(13):
            features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
            features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

        # RMS: 2
        rms = librosa.feature.rms(y=y)[0]
        features["rms_mean"] = np.mean(rms)
        features["rms_std"] = np.std(rms)

        # ZCR: 2
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        features["zcr_mean"] = np.mean(zcr)
        features["zcr_std"] = np.std(zcr)

        # Spectral: 2
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        flatness = librosa.feature.spectral_flatness(y=y)[0]

        features["spectral_centroid_mean"] = np.mean(centroid)
        features["spectral_flatness_mean"] = np.mean(flatness)

        # ---------- Praat / Parselmouth features ----------
        sound = parselmouth.Sound(file_path)

        praat_duration = min(sound.get_total_duration(), max_duration)
        sound = sound.extract_part(
            from_time=0,
            to_time=praat_duration,
            preserve_times=False
        )

        # Pitch / F0: 5
        pitch = call(sound, "To Pitch", 0.0, 75, 500)

        f0_mean = call(pitch, "Get mean", 0, 0, "Hertz")
        f0_std = call(pitch, "Get standard deviation", 0, 0, "Hertz")
        f0_min = call(pitch, "Get minimum", 0, 0, "Hertz", "Parabolic")
        f0_max = call(pitch, "Get maximum", 0, 0, "Hertz", "Parabolic")

        features["f0_mean"] = f0_mean
        features["f0_std"] = f0_std
        features["f0_min"] = f0_min
        features["f0_max"] = f0_max
        features["f0_range"] = f0_max - f0_min

        # Jitter + shimmer: 2
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)

        features["jitter"] = call(
            point_process,
            "Get jitter (local)",
            0, 0, 0.0001, 0.02, 1.3
        )

        features["shimmer"] = call(
            [sound, point_process],
            "Get shimmer (local)",
            0, 0, 0.0001, 0.02, 1.3, 1.6
        )

        # HNR: 1
        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        features["hnr"] = call(harmonicity, "Get mean", 0, 0)

        # Formants: F1, F2, F3 mean + std = 6
        formant = call(sound, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)

        time_points = np.linspace(0.05, praat_duration - 0.05, 20)

        for formant_num in [1, 2, 3]:
            values = []

            for t in time_points:
                value = call(
                    formant,
                    "Get value at time",
                    formant_num,
                    t,
                    "Hertz",
                    "Linear"
                )

                if not np.isnan(value):
                    values.append(value)

            if len(values) > 0:
                features[f"F{formant_num}_mean"] = np.mean(values)
                features[f"F{formant_num}_std"] = np.std(values)
            else:
                features[f"F{formant_num}_mean"] = np.nan
                features[f"F{formant_num}_std"] = np.nan

        return features

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [ ]:
from tqdm import tqdm
import pandas as pd

feature_rows = []
failed_files = []

SAVE_EVERY = 100

for idx, row in tqdm(df.iterrows(), total=len(df)):
    file_path = row[AUDIO_COL]

    features = extract_vowel_features(file_path)

    if features is not None:
        features["id"] = row["id"] if "id" in df.columns else idx
        features[LABEL_COL] = row[LABEL_COL]

        for col in ["age", "gender", "smoker", "cold_present", "cough_present", "fever_present"]:
            if col in df.columns:
                features[col] = row[col]

        feature_rows.append(features)

    else:
        failed_files.append({
            "index": idx,
            "path": file_path
        })


import os
os.makedirs(SAVE_DIR, exist_ok=True)

pd.DataFrame(feature_rows).to_csv(
    os.path.join(SAVE_DIR, "vowel_features_partial.csv"),
    index=False
)

pd.DataFrame(failed_files).to_csv(
    os.path.join(SAVE_DIR, "failed_vowel_files_partial.csv"),
    index=False
)

 14%|█▍        | 390/2777 [01:01<06:26,  6.18it/s]c:\Users\ishra\anaconda3\envs\ml\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1536
  warnings.warn(
100%|██████████| 2777/2777 [08:52<00:00,  5.22it/s]


In [50]:
SAVE_DIR = r"D:\first_iterations\ishrat-vowel-only"
vowel_features_df = pd.DataFrame(feature_rows)
failed_df = pd.DataFrame(failed_files)

vowel_features_df.to_csv(
    os.path.join(SAVE_DIR, "vowel_only_features.csv"),
    index=False
)

failed_df.to_csv(
    os.path.join(SAVE_DIR, "failed_vowel_files.csv"),
    index=False
)


In [51]:
#Ordering the csv

metadata_cols = [
    "id", "age", "gender", "smoker",
    "cold_present", "cough_present", "fever_present",
    "vowel_path", "disease"
]

metadata_cols = [col for col in metadata_cols if col in vowel_features_df.columns]

feature_cols = [col for col in vowel_features_df.columns if col not in metadata_cols]

vowel_features_df = vowel_features_df[metadata_cols + feature_cols]

vowel_features_df.head()

vowel_features_df.to_csv(
    os.path.join(SAVE_DIR, "vowel_only_features_ordered.csv"),
    index=False
)